# Experiment: Case-001 Clinician Response Label Capture Gate

Objective:
- Validate the May 23 public-safe clinician-response capture map.
- Confirm that doctor replies are reduced to labels before public repo updates.
- Keep adult mitapivat regulatory context separate from individual-fit claims.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "../../../data/workflows/case-001/2026-05-23-clinician-response-label-capture.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

## Structural Checks

The gate should be small enough to audit manually but strict enough to block raw private material, treatment instructions, and access/dosing actions from public Git updates.


In [ ]:
expected_steps = [
    "store_raw_reply_private_only",
    "extract_domain_answered",
    "classify_response_type",
    "map_missing_records_and_owner",
    "map_mitapivat_context_if_mentioned",
    "release_public_label_only",
]

actual_steps = [step["step_label"] for step in gate["capture_flow"]]
assert gate["checked_date"] == "2026-05-23"
assert gate["gate_label"] == "case001_clinician_response_label_capture_ready"
assert actual_steps == expected_steps
assert len(gate["source_checks"]) >= 9

required_step_fields = {"step_label", "private_action", "public_output", "stop_if"}
for step in gate["capture_flow"]:
    assert required_step_fields <= set(step)

actual_steps

## Privacy And Boundary Checks

Public fields must stay label-only. Blocked content must include identifiers, raw messages, patient-specific instructions, and access/dosing instructions.


In [ ]:
required_response_types = {
    "confirmed",
    "corrected",
    "not_confirmable_missing_records",
    "not_applicable",
    "under_specialist_review",
    "release_blocked_private_material",
    "patient_specific_instruction_private_only",
}
required_blocked = {
    "raw_records",
    "doctor_messages",
    "clinician_names",
    "hospital_names",
    "patient_specific_treatment_instructions",
    "dose_or_dosing_requests",
    "access_import_or_travel_instructions",
    "trial_screening_instruction",
}

assert required_response_types <= set(gate["allowed_response_types"])
assert required_blocked <= set(gate["blocked_public_content"])
assert "doctor_messages" not in gate["allowed_public_fields"]
assert "patient_specific_treatment_instructions" not in gate["allowed_public_fields"]

{
    "response_types": len(gate["allowed_response_types"]),
    "blocked_public_items": len(gate["blocked_public_content"]),
}

## Decision

The gate passes if a clinician answer changes the private work queue first and public documentation stays label-only. Adult mitapivat approvals are context for qualified review, not a case-specific conclusion.
